# Charge transport in an amorphous thiophene morphology

This tutorial applies the transport calculators of VOTCA-XTP to a real morphology: 1000 thiophene molecules from a LAMMPS simulation, with site energies, electronic couplings and reorganization energies already in the state file `state.hdf5` (the workflow that produces them is shown in the QMMM tutorials). We look at the energy landscape and the couplings, compute bulk mobilities of electrons and holes, and run a time-of-flight calculation that shows where the carriers are held up.

For the methods and their options on a controlled model system, see the tutorial *Charge and exciton transport on a model lattice* and the chapter *Transport Simulations* of the manual.

## Requirements
* VOTCA installed and the environment activated with `VOTCARC.bash` (see the [installation instructions](https://github.com/votca/votca/blob/master/share/sphinx/INSTALL.rst)).
* Python with `numpy` and `matplotlib`.

In Jupyter, `!` runs a shell command.

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt


def run(command):
    # run a VOTCA command quietly (raises if it fails)
    subprocess.run(command, shell=True, check=True, capture_output=True)


def results(filename):
    # read a 'key value unit' result file into a dictionary
    values = {}
    with open(filename) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            key, value = line.split()[:2]
            try:
                values[key] = float(value)
            except ValueError:
                values[key] = value
    return values

## Site energies and couplings

`eanalyze` writes histograms of the site energies and of the site energy differences of neighbouring pairs (here with a resolution of 0.03 eV); `ianalyze` writes the distribution of the couplings $\log_{10}|J|^2$. See [eanalyze](https://www.votca.org/xtp/eanalyze.html) and [ianalyze](https://www.votca.org/xtp/ianalyze.html) for the options.

In [ ]:
!xtp_run -e eanalyze -c resolution_sites=0.03 -f state.hdf5
!xtp_run -e ianalyze -c resolution_logJ2=0.1 states=e,h -f state.hdf5

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for carrier, label in [("e", "electrons"), ("h", "holes")]:
    hist = np.loadtxt(f"eanalyze.sitehist_{carrier}.out")
    ax1.plot(hist[:, 0] - np.average(hist[:, 0], weights=hist[:, 1]), hist[:, 1], label=label)
    coupling = np.loadtxt(f"ianalyze.ihist_{carrier}.out")
    ax2.plot(coupling[:, 0], coupling[:, 1], label=label)
ax1.set_xlabel("site energy - mean (eV)")
ax1.set_ylabel("sites")
ax1.legend()
ax2.set_xlabel(r"$\log_{10}(|J|^2/\mathrm{eV}^2)$")
ax2.set_ylabel("pairs")
ax2.legend()
plt.show()

The width of the site energy distribution (energetic disorder) controls how strongly carriers are trapped; the couplings span many orders of magnitude, so a few well-coupled pairs carry most of the current.

## Bulk mobility

`kmcmultiple` follows carriers in the periodic box. With one carrier, the exact single-carrier mobility from the master equation (printed with `exact`, the default) is the reference. Electrons drift against the field; the reported mobility is positive when carriers drift along the force on them. We use 8 independent runs of $10^7$ KMC events each.

In [ ]:
!xtp_run -e kmcmultiple -c carriertype=electron field="1e7 0 0" numberofcarriers=1 runs=8 runtime=10000000 outputtime=0 -f state.hdf5

In [ ]:
r = results("kmcmultiple_results.dat")
print(f"KMC {r['mobility']:.2e} +- {r['mobility_error']:.1e} cm^2/Vs, exact {r['exact_mobility']:.2e} cm^2/Vs")

The statistical error of the KMC is large: the carrier spends most of its time on a few low-energy molecules and leaves them rarely, so even $8\cdot10^7$ events contain few independent escapes. The exact value needs no sampling and is cheap; with a very short KMC part (`runtime=1000`) we use it to compare electrons and holes and to scan the field:

In [ ]:
fields = np.array([0.5, 1, 2, 5, 10, 20]) * 1e7  # V/m
for carrier in ["electron", "hole"]:
    mu = []
    for F in fields:
        run(f'xtp_run -e kmcmultiple -c carriertype={carrier} field="{F} 0 0" runtime=1000 outputtime=0 resultfile=scan.dat ratefile=x occfile=x -f state.hdf5')
        mu.append(results("scan.dat")["exact_mobility"])
    plt.semilogy(np.sqrt(fields / 1e2), mu, "o-", label=carrier + "s")
plt.xlabel(r"$\sqrt{F}$ ($\sqrt{\mathrm{V/cm}}$)")
plt.ylabel("bulk mobility (cm$^2$/Vs)")
plt.legend()
plt.show()

Electrons are about two orders of magnitude more mobile than holes in this morphology. The irregular field dependence reflects the few low-energy sites that dominate transport in such a small box.

## Time of flight and trap sites

For a time-of-flight calculation the periodicity is cut across the box vector `a`; electrons are injected in a 1 nm slab at one face and collected at the other (the box is only about 5 nm long, so this is a very thin "sample"). Electrons move against the field, so the field points in $-x$. `grw` gives the mean time of flight exactly and, in `grw_occupation.dat`, the expected time $g_A$ each molecule holds the electron before it reaches the sink, and the *fraction of the time of flight* $g_A/\langle T\rangle$ (adding up to 1 over all molecules). All output quantities are defined in the section *Output quantities* of the manual chapter *Transport Simulations*.

In [ ]:
!xtp_run -e grw -c carriertype=electron field="-1e7 0 0" -f state.hdf5

In [ ]:
grw = results("grw_results.dat")
occ = np.loadtxt("grw_occupation.dat", usecols=(0, 1, 2, 5))
segment, x, energy, fraction = occ.T
plt.scatter(x, energy, c=np.log10(np.maximum(fraction, 1e-8)), s=8, cmap="viridis")
plt.colorbar(label=r"$\log_{10}$ of the fraction of the time of flight $g_A/\langle T\rangle$")
plt.xlabel("position along the transport direction (nm)")
plt.ylabel("site energy (eV)")
plt.show()
order = np.argsort(fraction)[::-1][:5]
print(f"time of flight {grw['tau_mean']:.2e} s")
for s in order:
    print(f"segment {int(segment[s])}: x = {x[s]:.2f} nm, energy {energy[s]:.3f} eV, {fraction[s]:.0%} of the time of flight")

A handful of molecules hold the electron for most of the time of flight. These trap sites can be analysed further, e.g. by their conformation or environment; their identification from the occupation map needs no sampling.

`kmctof` simulates the same experiment; its mean transit time agrees with `grw` within the statistical error, and the distribution over runs shows how broad the transit times are:

In [ ]:
run('xtp_run -e kmctof -c carriertype=electron field="-1e7 0 0" runs=2000 -f state.hdf5')
tof = results("kmctof_results.dat")
print(f"KMC {tof['transit_mean']:.2e} +- {tof['transit_error']:.1e} s, grw {grw['tau_mean']:.2e} s")
transit = np.loadtxt("kmctof_runs.dat", usecols=5)
plt.hist(transit, bins=np.logspace(np.log10(transit.min()), np.log10(transit.max()), 40))
plt.xscale("log")
plt.xlabel("transit time (s)")
plt.ylabel("runs")
plt.show()

## Summary

* The energy landscape and the couplings of a morphology determine where carriers are trapped; `eanalyze` and `ianalyze` give a first impression.
* `kmcmultiple` gives bulk mobilities; its exact single-carrier values are cheap and free of statistical errors, and they are the reference for the KMC.
* `grw` and `kmctof` give times of flight; the occupation map of `grw` identifies the molecules that act as traps.

This morphology is small (1000 molecules, about 5 nm): mobilities from it are dominated by a few sites and should be taken as an illustration. Production calculations need larger systems and several snapshots.